# TradeFlow AI — nb2_chromadb_index

Generates the HS code ChromaDB vector index.
**Fix #5 (Production note)**: Untuk produksi nyata, ganti `embedding_function` default dengan Gemini Text Embedding (sesuai PRD §3: 'ChromaDB + Gemini reranker').


In [ ]:
!pip install -q chromadb sentence-transformers pandas


In [ ]:
import chromadb
import pandas as pd

client     = chromadb.PersistentClient(path='./chroma_db')
collection = client.get_or_create_collection(name='hs_codes')

# NOTE untuk produksi: ganti embedding_function di atas dengan:
# from chromadb.utils.embedding_functions import GoogleGenerativeAiEmbeddingFunction
# ef = GoogleGenerativeAiEmbeddingFunction(api_key=GEMINI_API_KEY, model_name='models/text-embedding-004')
# collection = client.get_or_create_collection(name='hs_codes', embedding_function=ef)

data = [
    # --- KUNCI JAWABAN dari Ground Truth v5.2 ---
    {'hs':'84821000','desc':'Bantalan peluru. Ball bearings. Bearings & bearing units.'},
    {'hs':'84822000','desc':'Bantalan rol tirus, termasuk rakitan kerucut dan rol tirus. Tapered roller bearings.'},
    {'hs':'84825000','desc':'Bantalan rol silinder lainnya. Other cylindrical roller bearings.'},
    {'hs':'84828000','desc':'Lain-lain, termasuk bantalan kombinasi peluru atau rol. Other combined bearings.'},
    {'hs':'28151110','desc':'Natrium hidroksida (soda kaustik) dalam bentuk padat. Caustic Soda Flakes. UN1813 Class 8 Dangerous Goods.'},
    {'hs':'72193590','desc':'Produk canai lincin dari baja tahan karat. Cold Rolled Stainless Steel Coil. Grade J3.'},
    # --- Data Pengecoh (Automotive) ---
    {'hs':'87070000','desc':'Bodi untuk kendaraan bermotor. Bodies for motor vehicles.'},
    {'hs':'87071010','desc':'Bodi Untuk gokart dan mobil golf. For go-karts and golf cars.'},
    {'hs':'87079021','desc':'Bodi Untuk mobil termasuk limousin. For motor cars including stretch limousines.'},
    {'hs':'87080000','desc':'Bagian dan aksesori kendaraan bermotor. Parts and accessories of motor vehicles.'},
    {'hs':'87082100','desc':'Sabuk pengaman. Safety seat belts.'},
    {'hs':'87082200','desc':'Kaca depan windshield, jendela belakang. Front windscreens windshields rear windows.'},
    # --- Data Tambahan (Chemical / Fertilizer) ---
    {'hs':'31021000','desc':'Urea mengandung lebih dari 45% nitrogen. Urea PRILLED UREA nitrogen fertilizer.'},
    {'hs':'39269099','desc':'Artikel lain dari plastik. Other articles of plastics.'},
    {'hs':'84713020','desc':'Mesin pengolah data otomatis portabel laptop. Portable automatic data processing machines laptops.'},
]

df = pd.DataFrame(data)
print(f'Indexing {len(df)} HS codes into ChromaDB...')
collection.add(
    documents=df['desc'].tolist(),
    metadatas=[{'hs_code': hs} for hs in df['hs']],
    ids=df['hs'].tolist()
)
print('Indexing complete. Database saved to ./chroma_db')
